# CookMatch — Colab Full Run

**Colab quirk:** Opening this from GitHub loads **this notebook file only**. Cell 1 clones fresh **code**, but **does not update the cells you see** in an already-open tab.

**After every push to GitHub:** open a **new** Colab tab from the link below (or open the copy inside the cloned repo — see cell 1 output).

**Direct link (use this after repo updates):**  
https://colab.research.google.com/github/YUV3571/cookmatch-recipe-recommender/blob/main/notebooks/03_colab_full_run.ipynb

Setup notebook: [`00_colab_github_setup.ipynb`](https://colab.research.google.com/github/YUV3571/cookmatch-recipe-recommender/blob/main/notebooks/00_colab_github_setup.ipynb)

Flow: GitHub setup → Kaggle auth → load Food.com → train → recommend → ablation.

In [ ]:
# 1) Connect GitHub repo to Colab (run every new session)
import os
import sys
import pathlib

REPO_DIR = "/content/cookmatch-recipe-recommender"
REPO_URL = "https://github.com/YUV3571/cookmatch-recipe-recommender.git"
ZIP_URL = "https://github.com/YUV3571/cookmatch-recipe-recommender/archive/refs/heads/main.zip"
RAW = "https://raw.githubusercontent.com/YUV3571/cookmatch-recipe-recommender/main"
NOTEBOOK_IN_REPO = f"{REPO_DIR}/notebooks/03_colab_full_run.ipynb"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

git_ok = os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}") == 0
if git_ok:
    print("Connected via git clone")
else:
    print("git failed — using zip fallback")
    !wget -q {ZIP_URL} -O /content/repo.zip
    !unzip -q /content/repo.zip -d /content
    !mv /content/cookmatch-recipe-recommender-main {REPO_DIR}

%cd {REPO_DIR}

if not pathlib.Path("src/data/loader.py").exists():
    !mkdir -p src/data
    !wget -q {RAW}/src/data/loader.py -O src/data/loader.py
    !wget -q {RAW}/src/data/__init__.py -O src/data/__init__.py

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import colab_init  # noqa: F401

from src.data.loader import build_eval_recipe_catalog, load_recipes
from src.recommend.stage3 import Stage3Recommender

repo_nb = pathlib.Path(NOTEBOOK_IN_REPO).read_text(encoding="utf-8")
if "FAST_ABLATION" not in repo_nb:
    print("WARNING: cloned repo notebook looks outdated — pull latest on GitHub first.")
else:
    print("Cloned repo has latest notebook.")

print("GitHub ↔ Colab OK | cwd:", os.getcwd())
print("\nIf cells below still show OLD code (no FAST_ABLATION), this tab is stale.")
print("Open the cloned notebook instead:")
print("  Files sidebar → cookmatch-recipe-recommender → notebooks → 03_colab_full_run.ipynb")
print("Or open a fresh tab:")
print("  https://colab.research.google.com/github/YUV3571/cookmatch-recipe-recommender/blob/main/notebooks/03_colab_full_run.ipynb")
!ls src/data/

In [ ]:
# 2) Install dependencies
!pip install -q kagglehub pandas numpy scipy pyarrow

In [ ]:
# 3) Kaggle authentication
import os

# Option A (recommended): Kaggle API token
# Kaggle → Settings → API → Create New Token → copy KGAT_... value
os.environ["KAGGLE_API_TOKEN"] = "KGAT_your_token_here"  # paste your token

# Option B: upload kaggle.json instead (comment out Option A first)
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

assert os.environ.get("KAGGLE_API_TOKEN", "").startswith("KGAT_"), "Set a valid KAGGLE_API_TOKEN"
print("Kaggle token configured.")

In [ ]:
# 4) Load dataset
import os
import sys
import time

REPO_DIR = "/content/cookmatch-recipe-recommender"
import colab_init
colab_init.bind(REPO_DIR)

# --- RUN MODE ---
FULL_CATALOG = True      # False = quick 5000-row smoke test
FAST_ABLATION = True     # 30k eval catalog + 100 users (~5–15 min). False = hours on 231k
EVAL_CATALOG_SIZE = 30_000
USER_SAMPLE = 100 if FAST_ABLATION else 500
RECIPE_LIMIT = None if FULL_CATALOG else 5000
# ----------------

import kagglehub
from src.data.loader import (
    build_eval_recipe_catalog,
    get_dataset_path,
    load_interaction_split,
    load_recipes,
)

dataset_path = get_dataset_path()
print("Dataset path:", dataset_path)

t0 = time.time()
recipes = load_recipes(
    nrows=RECIPE_LIMIT,
    columns=["id", "name", "ingredients", "minutes", "tags"],
)
train = load_interaction_split("train")
validation = load_interaction_split("validation")

recipes_eval = build_eval_recipe_catalog(
    recipes, train, validation, max_recipes=EVAL_CATALOG_SIZE
)
recipes_train = recipes_eval if FAST_ABLATION else recipes

print(f"recipes loaded: {len(recipes)} ({'FULL' if FULL_CATALOG else 'SAMPLE'})")
print(f"train/eval catalog: {len(recipes_train)} recipes")
print(f"train interactions: {len(train)}")
print(f"validation rows: {len(validation)}")
print(f"ablation users: {USER_SAMPLE}")
print(f"load time: {time.time() - t0:.1f}s")

In [ ]:
# 5) Train Stage 3 recommender + demo recommendations
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext
from src.recommend.stage3 import Stage3Recommender

recipes_train = globals().get("recipes_train", recipes)

from src.data.loader import filter_interactions_to_catalog

train_catalog = filter_interactions_to_catalog(train, recipes_train)
recommender = Stage3Recommender().fit(recipes_train, train_catalog)
print(f"trained on {len(recipes_train)} recipes, {len(train_catalog)} interactions")

profile = UserProfile(diet="vegan", allergens=["nuts", "dairy", "gluten"])
context = SessionContext(pantry=["tomato", "pasta", "garlic"], max_minutes=30, meal_intent="main")
known_user = int(train["user_id"].iloc[0])

recs = recommender.recommend(profile, context, user_id=known_user, top_n=5)
for rec in recs:
    print(f"{rec.final_score:.3f} | {rec.name}")
    print(f"  why: {rec.explanation}")

In [ ]:
# 6) Ablation eval (fast path — reuses cell 5 model, no second fit)
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from src.eval.offline_eval import run_ablation
import pandas as pd
import time

recipes_train = globals().get("recipes_train", recipes)
USER_SAMPLE = globals().get("USER_SAMPLE", 100)
TOP_K = 10

assert "recommender" in globals(), "Run cell 5 first (trains recommender)"

print(f"Running ablation on {len(recipes_train)} recipes, {USER_SAMPLE} users...")
print("(Reusing cell 5 model — expect ~5–15 min with FAST_ABLATION=True)")
t0 = time.time()

results = run_ablation(
    recipes=recipes_train,
    train_interactions=train,
    held_out=validation,
    user_sample_size=USER_SAMPLE,
    k=TOP_K,
    stage3=recommender,
)

print(f"Ablation done in {(time.time()-t0)/60:.1f} min")
display(results.sort_values(by=f"hit_rate@{TOP_K}", ascending=False))

results.to_csv("ablation_results.csv", index=False)
print("Saved ablation_results.csv")

## Full catalog run

Uncomment below only if you accept long runtimes:

```python
recipes_full = load_recipes(columns=["id", "name", "ingredients", "minutes", "tags"])
recommender = Stage3Recommender().fit(recipes_full, train)
results_full = run_ablation(recipes=recipes_full, train_interactions=train, held_out=validation, user_sample_size=500, k=10)
results_full.to_csv("ablation_full.csv", index=False)
```